**[🏠 Course Home](00_START_HERE.ipynb)** | [⬅️ Previous: Sheet 1 (Grid Approx)](01_frequentist_vs_grid_approximation.ipynb) | **Sheet 2 of 4: Rapid Prototyping** | [➡️ Next: Sheet 3 (MCMC Mechanics)](03_mcmc_mechanics_from_scratch.ipynb)

---

# Sheet 2: The Fast Prototyping Engine: Quadratic (Laplace) Approximation

In **Sheet 1**, we evaluated probabilities on a rigid grid. In this notebook, we explore **Quadratic Approximation** (`quap`), which fits a Gaussian bell curve directly to the posterior peak in **0.001 seconds** using optimization.

## Part 1: The Mathematics of Curvature & The Hessian Matrix
- Log-posterior near peak is parabolic: $\log P(\theta) \approx C - \frac{1}{2\sigma^2}(\theta - \mu)^2$.
- Mode (1st derivative = 0) $\to$ Maximum a Posteriori (MAP) estimate.
- 2nd derivatives $\to$ **Hessian Matrix ($H$)**.
- Posterior Variance-Covariance Matrix $\Sigma = (-H)^{-1}$.

In [ ]:
set.seed(42)
heights <- rnorm(50, mean = 172.5, sd = 8.0)

neg_log_posterior <- function(params) {
  mu    <- params[1]
  sigma <- params[2]
  if (sigma <= 0 || sigma >= 30) return(1e10)
  log_lik   <- sum(dnorm(heights, mean = mu, sd = sigma, log = TRUE))
  log_prior <- dnorm(mu, mean = 170, sd = 15, log = TRUE) + dunif(sigma, 0, 30, log = TRUE)
  return(-(log_lik + log_prior))
}

opt_fit <- optim(
  par     = c(mu = 170, sigma = 10),
  fn      = neg_log_posterior,
  hessian = TRUE,
  method  = "BFGS"
)

map_estimates <- opt_fit$par
vcov_matrix   <- solve(opt_fit$hessian)
std_errors    <- sqrt(diag(vcov_matrix))

cat("=== Laplace / Quadratic Approximation Results ===\n")
cat(sprintf("MAP Mean (mu):     %.2f  (SE: %.3f)\n", map_estimates["mu"], std_errors["mu"]))
cat(sprintf("MAP Sigma (sigma):  %.2f  (SE: %.3f)\n", map_estimates["sigma"], std_errors["sigma"]))
cat("\nVariance-Covariance Matrix:\n")
print(round(vcov_matrix, 5))

## Part 2: Sampling from the Multivariate Normal Posterior (`MASS::mvrnorm`)
Once we have MAP and $\Sigma$, we can draw 10,000 samples instantly.

In [ ]:
suppressPackageStartupMessages(library(MASS))
set.seed(42)
quap_samples <- mvrnorm(n = 1e4, mu = map_estimates, Sigma = vcov_matrix)

cat("=== Posterior Draws from Quadratic Approximation ===\n")
cat(sprintf("Posterior Mean (mu):     %.2f cm\n", mean(quap_samples[, "mu"])))
cat(sprintf("Posterior SD (mu):       %.3f cm\n", sd(quap_samples[, "mu"])))
cat(sprintf("95%% Credible Interval:   [%.2f, %.2f] cm\n", 
            quantile(quap_samples[, "mu"], 0.025), quantile(quap_samples[, "mu"], 0.975)))

# 2D Joint Posterior Plot
plot(quap_samples[1:1500, "mu"], quap_samples[1:1500, "sigma"], 
     col = rgb(0, 0, 0.8, 0.2), pch = 16, cex = 0.8, las = 1,
     main = "2D Joint Posterior Ellipse (mu vs sigma)",
     xlab = "Mean mu (cm)", ylab = "SD sigma (cm)")
points(map_estimates["mu"], map_estimates["sigma"], col = "red", pch = 19, cex = 1.5)
legend("topright", legend = c("Posterior Draws", "MAP Peak"), col = c("darkblue", "red"), pch = c(16, 19), bty = "n")

## Part 3: Production Modeling with `rethinking::quap`

In [ ]:
suppressPackageStartupMessages(library(rethinking))
dat <- list(h = heights)
m_quap <- quap(
  alist(
    h ~ dnorm(mu, exp(log_sigma)),
    mu ~ dnorm(170, 15),
    log_sigma ~ dnorm(2, 1)
  ),
  data = dat
)
precis(m_quap)

## Part 4: Hands-On Challenge Exercises

### Exercise 1: Extract Parameter Correlation
Using the variance-covariance matrix `vcov_matrix`, compute the correlation between $\mu$ and $\sigma$:
$$\text{Cor}(\mu, \sigma) = \frac{\text{Cov}(\mu, \sigma)}{\sqrt{\text{Var}(\mu) \cdot \text{Var}(\sigma)}}$$
*(Are $\mu$ and $\sigma$ correlated in a simple Gaussian model? Hint: In a symmetrical Gaussian likelihood, they are orthogonal/uncorrelated!)*

### Exercise 2: When Does `quap` Fail?
If sample size is $n = 3$, why might `quap` produce an inaccurate interval for $\sigma$? *(Hint: With $n=3$, the distribution of $\sigma$ is strongly right-skewed, but `quap` forces it to be a symmetric Gaussian).* 

---

**[🏠 Course Home](00_START_HERE.ipynb)** | [⬅️ Previous: Sheet 1 (Grid Approx)](01_frequentist_vs_grid_approximation.ipynb) | [➡️ Next: Sheet 3 (MCMC Mechanics)](03_mcmc_mechanics_from_scratch.ipynb)